# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll inspect the Record Sets, Fields, and Columns using the Croissant schema. All references use the `@id` of entities.

In [ ]:
# List all record sets and their fields by @id
from pprint import pprint

record_set_ids = []
print("Available Record Sets and their fields:")
for rs in dataset.metadata.record_sets:
    print(f"- Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) [{field.data_type}]")
    print("")
if not record_set_ids:
    print("  No RecordSets found. Please check dataset structure.")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis (using their `@id`s as previously listed).

In [ ]:
# Extract data from each record set by @id

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for Record Set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"  Failed to load records: {e}")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing numeric fields, grouping, and more using field `@id`s.

**Example:** Let's investigate the protein-level table (if available) and filter and normalize numeric fields such as peptide counts or molecular weights.

*Please update the `record_set_id`, `numeric_field_id`, and `group_field_id` variables below with appropriate `@id` values for your use case as listed in the overview above.*

In [ ]:
# Example EDA - adjust these @ids as appropriate for your schema overview output
# (Use the printed Record Set and Field @ids from earlier cells)

# For demonstration, we try the first record set and pick numeric/text fields by inspection
if record_set_ids:
    record_set_id = record_set_ids[0]  # update if multiple record sets exist
    df = dataframes[record_set_id]
    print(f"Working on DataFrame for {record_set_id} with columns: {df.columns.tolist()}")
    # Try to auto-pick a numeric field
    numeric_field_id = None
    for col in df.columns:
        # Try to guess numeric column by dtype or name
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
        elif 'peptide' in col.lower() or 'mw' in col.lower() or 'count' in col.lower():
            numeric_field_id = col
            break
    # Fallback
    if not numeric_field_id:
        numeric_field_id = df.columns[0]
    print(f"Using numeric field: {numeric_field_id}\n")

    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10

    try:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to auto-pick a group field (categorical/text)
        group_field_id = None
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    except Exception as e:
        print(f"Could not perform EDA: {e}")
else:
    print("No record sets found in dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Visualize the distribution of the selected numeric field
if record_set_ids and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If group_field_id selected, boxplot by group
    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.xticks(rotation=30)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2) FAIR² dataset using the `mlcroissant` library. We loaded metadata, inspected the structure by referencing all entities by their Croissant `@id`, extracted records, performed basic EDA and visualization, and demonstrated how to use the field and record set identifiers for reliable programmatic analysis.

<br>
**Key Takeaways:**

- The dataset is richly annotated using the Croissant schema, allowing transparent referencing of all entities via `@id`.
- Using `mlcroissant`, you can programmatically access and analyze experimental datasets in a reproducible way.
- The analyses shown here provide an entry point for deeper biological or biomedical inquiry into protein abundance and related data, and are easily adaptable to other Croissant-powered datasets.
